# Dataset 1: MeteoSwiss Live Temperature Visualization
This notebook downloads live instantaneous air temperature data from the MeteoSwiss network and visualizes it interactively.

In [ ]:
import pandas as pd
import plotly.express as px
from IPython.display import display

## 1. Load Live Data
Fetching the dataset directly from the opendata.swiss URL using the CSV endpoint.

In [ ]:
url = "https://data.geo.admin.ch/ch.meteoschweiz.messwerte-lufttemperatur-10min/ch.meteoschweiz.messwerte-lufttemperatur-10min_en.csv"

# The file uses semicolons as separators and standard encoding
try:
    df = pd.read_csv(url, sep=';', encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(url, sep=';', encoding='latin1')

# Clean up valid temperature mapping
temp_col = [col for col in df.columns if 'Temperature' in col][0]

df = df.dropna(subset=[temp_col, 'Latitude', 'Longitude'])
df[temp_col] = pd.to_numeric(df[temp_col], errors='coerce')
df = df.dropna(subset=[temp_col])

print(f"Loaded {len(df)} station measurements.")
display(df.head())

## 2. Interactive Map
Plotting the current temperatures mapped to their geographical coordinates using `plotly.express.scatter_mapbox`.

In [ ]:
fig_map = px.scatter_mapbox(
    df,
    lat="Latitude",
    lon="Longitude",
    color=temp_col,
    hover_name="Station",
    hover_data=["Canton", "Measurement height m a. sea level"],
    color_continuous_scale=px.colors.sequential.Turbo,
    size_max=15,
    zoom=6.5,
    center={"lat": 46.8, "lon": 8.2}, # Approximate center of Switzerland
    title="Live Air Temperature across Switzerland (SwissMetNet)",
    mapbox_style="carto-positron"
)

fig_map.update_traces(marker=dict(size=12))
fig_map.update_layout(height=650, margin=dict(r=0, t=50, l=0, b=0))
fig_map.show()

## 3. Top Extremes: Hottest and Coldest Stations
Visualising the top 10 hottest and coldest measurement points across the country in simple bar charts.

In [ ]:
# Find top 10 extremes
df_sorted = df.sort_values(by=temp_col)
df_coldest = df_sorted.head(10)
df_hottest = df_sorted.tail(10).sort_values(by=temp_col, ascending=False)

# Plotting Coldest
fig_cold = px.bar(
    df_coldest, 
    x="Station", 
    y=temp_col, 
    color=temp_col,
    title="Top 10 Coldest Stations",
    color_continuous_scale="Blues_r",
    text_auto='.1f'
)
fig_cold.update_layout(template="plotly_white")
fig_cold.show()

# Plotting Hottest
fig_hot = px.bar(
    df_hottest, 
    x="Station", 
    y=temp_col, 
    color=temp_col,
    title="Top 10 Hottest Stations",
    color_continuous_scale="Reds",
    text_auto='.1f'
)
fig_hot.update_layout(template="plotly_white")
fig_hot.show()